# CodAdapt 2.1.0 — actual Kaggle native qualification

Upload the external test ZIP as an authorized input. Set its path below. The
standalone notebook carries its trusted delivery SHA; do not replace it with the
hash of an unknown package. Internet/compiler availability are owner's settings.
No Git/account commands are run. Normal Linux auto inference stays Python; this
test selects native explicitly. All failures return a ZIP when the runner starts.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, platform, stat, subprocess, sys, uuid, zipfile
PACKAGE = Path('/kaggle/input/codadapt-native-test/CODADAPT_210_EXTERNAL_TEST_PACKAGE.zip')
EXPECTED_PACKAGE_SHA256 = 'PASTE_TRUSTED_DELIVERY_SHA256'
EXPECTED_PYTHON = '3.13.15'
# Optional actual Linux native wheel; leave None to build from source on Kaggle.
NATIVE_WHEEL = None
NATIVE_WHEEL_SHA256 = None
assert platform.system() == 'Linux' and Path('/kaggle/working').is_dir()
WORK = Path('/kaggle/working') / ('codadapt210_' + uuid.uuid4().hex[:10])


In [ ]:
def safe_extract(archive,destination,expected_sha256):
    archive=Path(archive);destination=Path(destination).resolve()
    if hashlib.sha256(archive.read_bytes()).hexdigest()!=expected_sha256:raise ValueError('Untrusted package SHA256')
    if destination.exists():raise ValueError('Extraction destination must be new')
    with zipfile.ZipFile(archive) as z:
        seen=set();members=[];total=0
        for item in z.infolist():
            name=item.filename;parts=name.rstrip('/').split('/')
            if '\x00' in item.orig_filename or item.orig_filename!=name:raise ValueError('Noncanonical ZIP filename')
            if not name or ':' in name or '\\' in name or PurePosixPath(name).is_absolute() or any(p in ('','.','..') for p in parts):raise ValueError('Unsafe ZIP name')
            if stat.S_ISLNK(item.external_attr>>16) or name.casefold() in seen:raise ValueError('ZIP alias or link')
            seen.add(name.casefold());total+=item.file_size
            if total>150*1024**2:raise ValueError('ZIP expansion budget exceeded')
            target=destination.joinpath(*parts).resolve()
            if not target.is_relative_to(destination):raise ValueError('ZIP escape')
            members.append((item,target))
        destination.mkdir(parents=True)
        for item,target in members:
            if item.is_dir():target.mkdir(parents=True,exist_ok=True)
            else:target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(item))
    return destination

BOOTSTRAP_OK = False
try:
    extracted = safe_extract(PACKAGE, WORK / 'package', EXPECTED_PACKAGE_SHA256)
    manifest = json.loads((extracted / 'PACKAGE_MANIFEST.json').read_text())
    for name, row in manifest['files'].items():
        path = (extracted / name).resolve()
        assert path.is_relative_to(extracted.resolve())
        assert hashlib.sha256(path.read_bytes()).hexdigest() == row['SHA256'], name
    print('Package SHA256 and every bound payload verified before execution.')
    BOOTSTRAP_OK = True
except Exception as error:
    WORK.mkdir(parents=True, exist_ok=True)
    status = {'status': 'HOLD_BOOTSTRAP_FAILED', 'error': repr(error), 'native_calls': 0, 'source_executed': False}
    failure = WORK / 'BOOTSTRAP_STATUS.json'
    failure.write_text(json.dumps(status, indent=2))
    archive = WORK / 'CODADAPT_210_EXTERNAL_RESULTS.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(failure, failure.name)
    print('Bootstrap failed; return this ZIP:', archive)
    raise RuntimeError('Package verification failed; no package code was executed') from error


In [ ]:
env = dict(os.environ, NETRC=os.devnull, PIP_CONFIG_FILE=os.devnull)
import importlib.util
if importlib.util.find_spec('psutil') is None:
    subprocess.run([sys.executable, '-I', '-B', '-m', 'pip', '--isolated', '--no-input', '--keyring-provider', 'disabled', 'install', '--index-url', 'https://pypi.org/simple', '--only-binary=:all:', 'psutil==7.2.2'], env=env, check=True)
repository = extracted / 'repository'
output = WORK / 'results'
command = [sys.executable, '-I', '-B', str(repository / 'tools/run_external.py'), '--context', 'kaggle', '--expected-python', EXPECTED_PYTHON, '--output', str(output), '--allow-public-index']
if NATIVE_WHEEL is not None:
    assert NATIVE_WHEEL_SHA256 is not None
    command += ['--native-wheel', str(NATIVE_WHEEL), '--expected-wheel-sha256', NATIVE_WHEEL_SHA256, '--sdist', str(extracted / 'sources/codadapt-2.1.0.tar.gz')]
result = subprocess.run(command, cwd=repository, env=env, timeout=1900)
print('Exit status:', result.returncode)


In [ ]:
report = output / 'FINAL_STATUS.json'
if report.exists():
    print(json.dumps(json.loads(report.read_text()), indent=2))
archive = output / 'CODADAPT_210_EXTERNAL_RESULTS.zip'
assert archive.is_file(), 'Runner did not start: retain the preceding notebook error.'
print('Return this entire file:', archive)
print('SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
from IPython.display import FileLink, display
display(FileLink(str(archive)))


Return the entire results ZIP, not only a PASS screenshot. It includes failed
stages, native counters, precise versions, tests, model outputs, metrics and built
Linux wheel with SHA. A generic Linux job does not qualify Kaggle. No local tool
test substitutes for this run; automatic dispatch/migration remains separately gated.

If FileLink is unavailable, download the ZIP from Kaggle Outputs under /kaggle/working.
